# Step 6 · Order of magnitude: the LTS/MLCC curve vs a simple "km of cycle lanes" model

**Question:** is the bike-share gain predicted by our network model (steps 3–5) of the same order of magnitude
as a classic, much simpler model in which the share grows with the **km of cycle lanes built**?

The simple model (a *first-order* model, from colleagues' work) is

$$P_b(t)=P_b(0)+(P_b^\infty-P_b(0))\left(1-e^{-k\,u}\right)$$

where $u$ = new km of *piste + bande cyclable* (French statistics: separated tracks + painted lanes), $k$ a
city-specific efficiency and $P_b^\infty$ the potential share; one km costs about 400 €/m.

To compare the two on the **same axes**, every LTS edge also receives a French facility class (piste, bande,
voie verte, calm zone, …) read from its OSM tags — the LTS itself is not changed. Then:

1. observed shares (INSEE) and the year Grenoble had the same share as Paris 2023 (11 %);
2. piste + bande km vs LTS km in 2017–2022 and 2026;
3. calibration of $k$ for Grenoble and Paris; 4. transfer of $k$ between cities (the "ρ rule");
5. replay of the step-5 MLCC lookup, logging **which** streets are upgraded at each budget step;
6. share vs budget and share vs new piste-equivalent km for both models;
7. the equivalent $k$ of the MLCC curve; 8. **order-of-magnitude table**: budget and km needed for +2.5, +5,
   +10 percentage points; 9. harmonised comparison (only the lane-driven part of the growth).

| | |
|---|---|
| **Input** | Grenoble networks (step 1), Grenoble INSEE shares (step 2), MLCC lookup (step 5) |
| **Output** | `results/order_of_magnitude/` — in particular `order_of_magnitude_summary.csv` |
| **Run time** | ~5 s if `mlcc_upgrade_log.csv` exists, ~5 min otherwise |

> `book.tex` and `bicycle_model.tex` are the colleague's model documents (first-order bike-share model, backcasting chapter). They are cited for the equations but are **not included** in this public repository.

In [1]:
# Setup: find the repository root and make the `cyclelts` package importable.
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "cyclelts").is_dir())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from pathlib import Path
import json, pickle, math
import numpy as np, pandas as pd, networkx as nx
import plotly.graph_objects as go
from scipy.optimize import curve_fit
from IPython.display import display

from cyclelts import config
LTS_DIR = config.run_dir('Grenoble')
INSEE_CSV = config.INSEE_SHARES_DIR / 'grenoble_38185_modal_shares_2006_2022.csv'
LOOKUP_DIR = config.LOOKUP_DIR
OUT = config.ORDER_OF_MAGNITUDE_DIR
OUT.mkdir(parents=True, exist_ok=True)
YEARS = [2017, 2018, 2019, 2020, 2021, 2022, 2026]

# ---- Colleague's model parameters (bicycle_model.tex, table "Sensibilité des coûts d'abattement") ----
COST_EUR_PER_M = 400.0      # c^d. Interpreted as EUR per metre of lane (= 0.4 M EUR/km), see note in section 6
SCENARIOS = {               # alpha = share of growth due to lanes, P_inf = cycling potential
    'Conservative': dict(alpha=0.17, P_inf=0.14),
    'Intermediate': dict(alpha=0.21, P_inf=0.35),
    'Optimistic':   dict(alpha=0.25, P_inf=0.50),
}
# Paris reference (bicycle_model.tex, "Comparison to the work from LAET"): 386 -> 516 km, 3 % -> 11 %, 2018 -> 2023
PARIS = dict(L0=386.0, LT=516.0, P0=0.03, PT=0.11)

# ---- Values sent by the colleague (k in 1/metre, u_b = piste + bande length in metres) ----
P_INF = 0.50                                   # "for consistency, P_b^inf = 0.5"
K_PARIS = 2.3128785365068954e-07               # 1/m
K_FRANCE = 2.3723161632375222e-09              # 1/m
L_PARIS_2023_M = 515_984.61                    # pistes + bandes, Paris 2023 (open data "Linéaires d'aménagement cyclable")
P_PARIS_2023 = 0.11                            # Institut Paris Region, "La mobilité des Franciliens étudiée par leurs traces GPS"
REFERENCE_11PCT = dict(name='Paris 2023', share=P_PARIS_2023, L_km=L_PARIS_2023_M / 1000, k_per_km=K_PARIS * 1000)

# Grenoble piste+bande km around 2010 is NOT available from OSM (Overpass history starts in Sept. 2012).
# None -> linear back-extrapolation of the 2017-2022 OSM trend. Replace with an official value if you have one.
GRENOBLE_PB_2010_KM = None

## 1. Observed bicycle mode share (INSEE, commuting, commune of Grenoble)

INSEE separates *vélo* from *deux-roues motorisés* only from 2017. Before 2017 only "Deux-roues total" exists, so the
≈ 11 % seen by eye in 2009–2010 **includes motorbikes**. A bike-only estimate is obtained by subtracting the average
motorbike share of 2017–2019 (≈ 0.75 pp, very stable).

In [2]:
insee = pd.read_csv(INSEE_CSV, encoding='utf-8-sig')
moto_mean = insee.loc[insee.Year.between(2017, 2019), 'Deux-roues motorisé'].mean()
insee['bike_share_pct'] = insee['Vélo'].where(insee['Vélo'].notna(), insee['Deux-roues total'] - moto_mean)
insee['bike_is_estimate'] = insee['Vélo'].isna()
display(insee[['Year', 'Deux-roues total', 'Vélo', 'bike_share_pct', 'bike_is_estimate']].round(2))
print(f'Average motorbike share 2017-2019: {moto_mean:.2f} pp')

ref_pct = 100 * REFERENCE_11PCT['share']
same = insee.iloc[(insee.bike_share_pct - ref_pct).abs().argsort()[:1]]
YEAR_SAME = int(same.Year.iloc[0])
print(f'Grenoble year closest to {ref_pct:.0f}% bike share: {YEAR_SAME} '
      f'({same.bike_share_pct.iloc[0]:.2f}% bike-only estimate, {same["Deux-roues total"].iloc[0]:.2f}% two-wheelers)')

fig = go.Figure()
fig.add_scatter(x=insee.Year, y=insee['Deux-roues total'], name='Two-wheelers total (INSEE)', mode='lines+markers')
fig.add_scatter(x=insee.Year, y=insee.bike_share_pct, name='Bicycle (INSEE ≥2017, estimate <2017)', mode='lines+markers')
fig.add_hline(y=ref_pct, line_dash='dot', annotation_text=f"{REFERENCE_11PCT['name']} share")
fig.update_layout(template='plotly_white', height=420, title='Grenoble commuting mode share',
                  yaxis_title='%', xaxis_title='Year')
fig.show()

,Year,Deux-roues total,Vélo,bike_share_pct,bike_is_estimate
0,2006,9.56,NaN,8.81,True
1,2007,10.37,NaN,9.62,True
2,2008,10.62,NaN,9.87,True
3,2009,10.91,NaN,10.16,True
4,2010,11.27,NaN,10.52,True
5,2011,12.21,NaN,11.46,True
6,2012,12.79,NaN,12.04,True
7,2013,13.80,NaN,13.04,True
8,2014,15.00,NaN,14.25,True
9,2015,15.69,NaN,14.94,True


Average motorbike share 2017-2019: 0.75 pp
Grenoble year closest to 11% bike share: 2011 (11.46% bike-only estimate, 12.21% two-wheelers)


## 2. French facility class of every LTS edge

The LTS pipeline keeps a `facility_type` for each edge (derived from the OSM tags). It is mapped to the categories used in
French statistics. **The LTS value of each edge is not touched.**

| class | facility_type (LTS pipeline) | counted by colleague? |
|---|---|---|
| piste | dedicated cycleway, separated bike lane, sidepath/separated path, segregated shared-use path | yes |
| bande | painted bike lane | yes |
| voie verte | shared-use path (non-segregated) | no (variant) |
| zone apaisée | pedestrian zone with bicycle access, shared street, bicycle street | no |
| partage / couloir bus | shared bus lane, shared-lane marking, bikeable shoulder | no |
| trafic mixte | mixed traffic (residential, 30 km/h streets…) | no |
| futur (LTS 5) | 2026 candidate roads without cycling facility | no |

In [3]:
FACILITY_CLASS = {
    'dedicated cycleway': 'piste', 'separated bike lane': 'piste',
    'sidepath/separated path': 'piste', 'segregated shared-use path': 'piste',
    'painted bike lane': 'bande',
    'shared-use path': 'voie verte',
    'pedestrian zone with bicycle access': 'zone apaisée', 'shared street': 'zone apaisée',
    'bicycle street': 'zone apaisée',
    'shared bus lane': 'partage/bus', 'shared-lane marking': 'partage/bus', 'bikeable shoulder': 'partage/bus',
    'mixed traffic': 'trafic mixte',
}
CLASS_ORDER = ['piste', 'bande', 'voie verte', 'zone apaisée', 'partage/bus', 'trafic mixte', 'futur (LTS 5)']

def lts_of(d):
    return int(d.get('key', d.get('lts')))

def km_of(d):
    return float(d.get('distance', d.get('length', 0.0))) / 1000.0

def facility_class(d):
    if lts_of(d) == 5:
        return 'futur (LTS 5)'
    return FACILITY_CLASS.get(str(d.get('facility_type')), 'trafic mixte')

def is_piste_bande(d):
    return facility_class(d) in ('piste', 'bande')

def load_year(y):
    with (LTS_DIR / str(y) / f'grenoble_lts_{y}_SIMPLE.pkl').open('rb') as f:
        return pickle.load(f)   # trusted local pickle produced by the LTS notebooks

recs = []
for y in YEARS:
    G = load_year(y)
    for _, _, d in G.edges(data=True):
        recs.append(dict(year=y, lts=lts_of(d), facility=facility_class(d), km=km_of(d)))
edges_df = pd.DataFrame(recs)

cross_2026 = edges_df[edges_df.year == 2026].pivot_table(index='facility', columns='lts', values='km',
                                                         aggfunc='sum', fill_value=0).reindex(CLASS_ORDER).fillna(0)
cross_2026['total'] = cross_2026.sum(axis=1)
print('2026: km by French facility class (rows) and LTS (columns)')
display(cross_2026.round(1))

by_year = edges_df.pivot_table(index='year', columns='facility', values='km', aggfunc='sum', fill_value=0)
by_year = by_year.reindex(columns=[c for c in CLASS_ORDER if c in by_year.columns]).fillna(0)
lts_year = edges_df.pivot_table(index='year', columns='lts', values='km', aggfunc='sum', fill_value=0)
summary = pd.DataFrame({
    'piste_km': by_year['piste'], 'bande_km': by_year['bande'],
    'piste_bande_km': by_year['piste'] + by_year['bande'],
    'piste_bande_voie_verte_km': by_year['piste'] + by_year['bande'] + by_year['voie verte'],
    'lts1_km': lts_year[1], 'lts1_2_km': lts_year[1] + lts_year[2],
    'lts1_4_km': lts_year[[1, 2, 3, 4]].sum(axis=1),
})
summary['piste_bande_over_lts12'] = summary.piste_bande_km / summary.lts1_2_km
summary = summary.join(insee.set_index('Year')['bike_share_pct'])
display(summary.round(3))
summary.to_csv(OUT / 'grenoble_piste_bande_vs_lts_by_year.csv')
cross_2026.to_csv(OUT / 'grenoble_2026_facility_class_by_lts.csv')

fig = go.Figure()
for col, name in [('piste_bande_km', 'Piste + bande (colleague)'), ('lts1_km', 'LTS 1'),
                  ('lts1_2_km', 'LTS 1 + 2'), ('lts1_4_km', 'LTS 1–4 (all cycling)')]:
    fig.add_scatter(x=summary.index, y=summary[col], mode='lines+markers', name=name)
fig.update_layout(template='plotly_white', height=430, title='Grenoble — network length by definition',
                  yaxis_title='km (edge length, one count per street axis)', xaxis_title='Year')
fig.show()

2026: km by French facility class (rows) and LTS (columns)


lts,1,2,3,4,5,total
facility,,,,,,
piste,67.2,0.0,0.0,0.0,0.0,67.2
bande,0.0,43.2,0.5,0.0,0.0,43.7
voie verte,0.0,15.5,0.0,0.0,0.0,15.5
zone apaisée,36.7,1.1,0.0,0.0,0.0,37.8
partage/bus,0.0,12.4,9.0,0.3,0.0,21.7
trafic mixte,85.7,125.1,20.1,1.4,0.0,232.3
futur (LTS 5),0.0,0.0,0.0,0.0,32.6,32.6


,piste_km,bande_km,piste_bande_km,piste_bande_voie_verte_km,lts1_km,lts1_2_km,lts1_4_km,piste_bande_over_lts12,bike_share_pct
year,,,,,,,,,
2017,36.416,45.153,81.569,96.481,127.603,342.812,389.035,0.238,16.385
2018,35.964,45.939,81.903,96.751,131.939,345.579,392.700,0.237,17.480
2019,40.722,43.608,84.330,99.144,142.678,355.490,403.698,0.237,17.449
2020,47.503,45.985,93.488,106.096,148.566,360.540,405.040,0.259,18.721
2021,48.281,48.073,96.354,108.982,156.551,367.709,413.489,0.262,21.441
2022,46.881,47.809,94.690,107.166,161.031,375.060,415.449,0.252,22.711
2026,67.174,43.694,110.868,126.369,189.606,386.971,418.262,0.287,NaN


**Reading.** Piste + bande is only about a quarter of LTS 1 + 2: most LTS 1–2 km are calm mixed-traffic
streets, pedestrian zones and voies vertes, which the national statistic does not count. Also, bandes are LTS 2 (not 1),
and some LTS 1 km (zones apaisées) are not a facility at all. So *piste + bande ≠ LTS 1 + LTS 2*, and the right common
variable is the **piste + bande km**, computed on the same OSM graph.

## 3. Calibration of $k$

Two formulas exist in the documents:

* `book.tex` (other factors explain $1-\alpha$ of the growth):
  $k=\dfrac{-\ln\left(\frac{P^\infty-P(T)}{P^\infty-(1-\alpha)P(T)-\alpha P(0)}\right)}{\int u}$
* `bicycle_model.tex` (isolated effect): $k=\dfrac{-\ln\left(1-\alpha\frac{P(T)-P(0)}{P^\infty-P(0)}\right)}{\int u}$

$\int u$ = added piste + bande km. Since $k$ depends on the size of the region (`book.tex`), the dimensionless product
$\kappa = k\cdot L$ (L = piste + bande km at the end of the calibration period) is also reported: it is what can be
compared between Paris and Grenoble.

In [4]:
def k_book(P0, PT, du, alpha, P_inf):
    x = (P_inf - PT) / (P_inf - (1 - alpha) * PT - alpha * P0)
    return -math.log(x) / du if 0 < x < 1 and du > 0 else np.nan

def k_isolated(P0, PT, du, alpha, P_inf):
    x = 1 - alpha * (PT - P0) / (P_inf - P0)
    return -math.log(x) / du if 0 < x < 1 and du > 0 else np.nan

G_P0 = summary.loc[2017, 'bike_share_pct'] / 100
G_PT = summary.loc[2022, 'bike_share_pct'] / 100
G_L0, G_LT = summary.loc[2017, 'piste_bande_km'], summary.loc[2022, 'piste_bande_km']
G_DU = G_LT - G_L0
print(f'Grenoble 2017->2022: share {100*G_P0:.2f}% -> {100*G_PT:.2f}%, piste+bande {G_L0:.1f} -> {G_LT:.1f} km '
      f'(+{G_DU:.1f} km, +{100*G_DU/G_L0:.0f}%)')
print(f'Paris 2018->2023  : share {100*PARIS["P0"]:.0f}% -> {100*PARIS["PT"]:.0f}%, lanes {PARIS["L0"]:.0f} -> '
      f'{PARIS["LT"]:.0f} km (+{PARIS["LT"]-PARIS["L0"]:.0f} km, +{100*(PARIS["LT"]/PARIS["L0"]-1):.0f}%)')

rows = []
for scen, p in SCENARIOS.items():
    for region, (P0, PT, L0, LT) in {'Grenoble': (G_P0, G_PT, G_L0, G_LT),
                                     'Paris': (PARIS['P0'], PARIS['PT'], PARIS['L0'], PARIS['LT'])}.items():
        for formula, fn in [('book.tex', k_book), ('bicycle_model.tex', k_isolated)]:
            k = fn(P0, PT, LT - L0, p['alpha'], p['P_inf'])
            rows.append(dict(scenario=scen, region=region, formula=formula, alpha=p['alpha'], P_inf=p['P_inf'],
                             k_per_km=k, kappa_kL=k * LT,
                             note='' if np.isfinite(k) else f'P_inf below observed share ({100*PT:.1f}%)'))
k_table = pd.DataFrame(rows)
display(k_table.pivot_table(index=['scenario', 'formula'], columns='region',
                            values=['k_per_km', 'kappa_kL']).round(5))
display(k_table[k_table.note != ''][['scenario', 'region', 'formula', 'note']].drop_duplicates())
k_table.to_csv(OUT / 'k_calibration_grenoble_paris.csv', index=False)

Grenoble 2017->2022: share 16.38% -> 22.71%, piste+bande 81.6 -> 94.7 km (+13.1 km, +16%)
Paris 2018->2023  : share 3% -> 11%, lanes 386 -> 516 km (+130 km, +34%)


k_per_km          kappa_kL         
region                         Grenoble    Paris Grenoble    Paris
scenario     formula                                              
Conservative bicycle_model.tex      NaN  0.00102      NaN  0.52384
             book.tex               NaN  0.00288      NaN  1.48394
Intermediate bicycle_model.tex  0.00564  0.00041  0.53430  0.21405
             book.tex           0.00782  0.00052  0.74075  0.26855
Optimistic   bicycle_model.tex  0.00367  0.00033  0.34776  0.17260
             book.tex           0.00429  0.00038  0.40655  0.19850

,scenario,region,formula,note
0,Conservative,Grenoble,book.tex,P_inf below observed share (22.7%)
1,Conservative,Grenoble,bicycle_model.tex,P_inf below observed share (22.7%)


## 4. Colleague's ρ rule: $k^{Grenoble}=\rho\,k^{Paris}$

From the colleague's slide: if two regions grow from 0 to the **same** share, the first-order model gives
$k^{A}u^{A}=k^{B}u^{B}$, hence $k^{Grenoble}=\rho\,k^{Paris}$ with
$$\rho=\frac{u^{Paris}_b}{u^{Grenoble}_b}\quad(u_b = \text{piste + bande length when both had the same share}).$$

Three estimates are computed (all with $P_b^\infty = 0.5$, $u$ in metres):

* **(a) strict ρ rule at 11 %**: Paris 2023 (11 %, 515 984.61 m) ↔ Grenoble when its bike share was 11 %
  (≈ 2010–2011, interpolated). Grenoble's length at that date is extrapolated (no OSM history before 2012).
* **(b) generalised ρ (proposed)**: the integrated model from $P=0$ gives $k\,L(t) = f(P(t))$ with
  $f(P)=-\ln\!\left(1-\alpha_P P/P^\infty\right)$. Dividing Grenoble by Paris,
  $$k^{Gre}=k^{Paris}\,\frac{L^{Paris}}{L^{Gre}(t)}\,\frac{f(P^{Gre}(t))}{f(P^{Paris})}$$
  which reduces to (a) when the shares are equal, but can use **every OSM year 2017–2022 without extrapolation**.
  $\alpha_P$ is the value that makes $k^{Paris}L^{Paris}=f(0.11)$ (the same convention for both cities, so its effect
  mostly cancels).
* **(c) direct**: Grenoble's own 2017→2022 growth with the `bicycle_model.tex` isolated formula and the α implied by
  $k^{Paris}$ (see check below).

Checks on the colleague's numbers: $k^{Paris}L^{Paris}$, the national length implied by $k^{France}$, and the α that
reproduces $k^{Paris}$ from Paris 3 % → 11 % (2018 → 2023, 386 → 516 km).

In [5]:
kL_paris = K_PARIS * L_PARIS_2023_M
u_france_km = kL_paris / K_FRANCE / 1000
alpha_paris_step = (1 - math.exp(-K_PARIS * (L_PARIS_2023_M - 1000 * PARIS['L0']))) * (P_INF - PARIS['P0']) / (PARIS['PT'] - PARIS['P0'])
ALPHA_FROM_ZERO = (1 - math.exp(-kL_paris)) * P_INF / P_PARIS_2023
print(f'k_Paris * L_Paris(2023)               = {kL_paris:.4f}')
print(f'rho_France = k_France / k_Paris        = {K_FRANCE / K_PARIS:.5f}  -> implied national piste+bande = {u_france_km:,.0f} km')
print(f'alpha reproducing k_Paris, 3%->11% with +{L_PARIS_2023_M/1000-PARIS["L0"]:.0f} km (isolated formula): {alpha_paris_step:.3f}')
print(f'alpha_P used in the generalised rule (from P=0 to 11% with the whole network): {ALPHA_FROM_ZERO:.3f}')
ALPHA_COLLEAGUE = alpha_paris_step

def f_share(P, alpha=ALPHA_FROM_ZERO, P_inf=P_INF):
    return -np.log(1 - alpha * np.asarray(P, float) / P_inf)

# (a) strict rule at 11 %
pb = summary.loc[[y for y in summary.index if y <= 2022], 'piste_bande_km']
slope, intercept = np.polyfit(pb.index.values.astype(float), pb.values, 1)
ins = insee.sort_values('Year')
year_11 = float(np.interp(100 * P_PARIS_2023, ins.bike_share_pct, ins.Year))
L_G_same = GRENOBLE_PB_2010_KM if GRENOBLE_PB_2010_KM is not None else slope * year_11 + intercept
src = 'user value' if GRENOBLE_PB_2010_KM is not None else f'OSM 2017-2022 trend ({slope:+.2f} km/yr) extrapolated'
rho_strict = L_PARIS_2023_M / (1000 * L_G_same)
k_strict = rho_strict * K_PARIS
print(f'\n(a) Grenoble reached 11% bike share in ~{year_11:.1f}; piste+bande then ~{L_G_same:.1f} km ({src})')
print(f'    rho = {rho_strict:.2f}  ->  k_Grenoble = {k_strict:.3e} 1/m')

# (b) generalised rule, every OSM year
gen = summary.loc[[y for y in summary.index if y <= 2022], ['bike_share_pct', 'piste_bande_km', 'piste_bande_voie_verte_km']].copy()
gen['rho'] = L_PARIS_2023_M / (1000 * gen.piste_bande_km)
gen['share_factor'] = f_share(gen.bike_share_pct / 100) / f_share(P_PARIS_2023)
gen['k_grenoble_per_m'] = K_PARIS * gen.rho * gen.share_factor
gen['k_grenoble_per_m_with_voie_verte'] = K_PARIS * L_PARIS_2023_M / (1000 * gen.piste_bande_voie_verte_km) * gen.share_factor
display(gen.style.format({'bike_share_pct': '{:.2f}', 'piste_bande_km': '{:.1f}', 'piste_bande_voie_verte_km': '{:.1f}',
                          'rho': '{:.2f}', 'share_factor': '{:.3f}', 'k_grenoble_per_m': '{:.3e}',
                          'k_grenoble_per_m_with_voie_verte': '{:.3e}'}))
k_gen = float(gen.k_grenoble_per_m.mean())
print(f'(b) generalised rho, mean 2017-2022: k_Grenoble = {k_gen:.3e} 1/m '
      f'(range {gen.k_grenoble_per_m.min():.2e} - {gen.k_grenoble_per_m.max():.2e})')

# (c) direct calibration on Grenoble 2017 -> 2022 with the colleague's alpha
k_direct = k_isolated(G_P0, G_PT, 1000 * G_DU, ALPHA_COLLEAGUE, P_INF)
print(f'(c) direct 2017->2022 (alpha={ALPHA_COLLEAGUE:.2f}, +{1000*G_DU:,.0f} m): k_Grenoble = {k_direct:.3e} 1/m')

COLLEAGUE_K = {          # 1/km, all with P_inf = 0.5
    'ρ rule, 11 % (≈2010)': 1000 * k_strict,
    'generalised ρ (OSM 2017–22)': 1000 * k_gen,
    'direct Grenoble 2017–22': 1000 * k_direct,
}
transfer = pd.DataFrame([dict(method=m, k_per_m=k / 1000, k_per_km=k, ratio_to_k_Paris=k / 1000 / K_PARIS,
                              ratio_to_k_France=k / 1000 / K_FRANCE) for m, k in COLLEAGUE_K.items()])
display(transfer.style.format({'k_per_m': '{:.3e}', 'k_per_km': '{:.5f}', 'ratio_to_k_Paris': '{:.1f}',
                               'ratio_to_k_France': '{:.0f}'}))
transfer.to_csv(OUT / 'k_grenoble_rho_rule.csv', index=False)
gen.to_csv(OUT / 'k_grenoble_generalised_rho_by_year.csv')

k_Paris * L_Paris(2023)               = 0.1193
rho_France = k_France / k_Paris        = 0.01026  -> implied national piste+bande = 50,306 km
alpha reproducing k_Paris, 3%->11% with +130 km (isolated formula): 0.174
alpha_P used in the generalised rule (from P=0 to 11% with the whole network): 0.511

(a) Grenoble reached 11% bike share in ~2010.5; piste+bande then ~58.4 km (OSM 2017-2022 trend (+3.37 km/yr) extrapolated)
    rho = 8.84  ->  k_Grenoble = 2.044e-06 1/m


,bike_share_pct,piste_bande_km,piste_bande_voie_verte_km,rho,share_factor,k_grenoble_per_m,k_grenoble_per_m_with_voie_verte
year,,,,,,,
2017,16.38,81.6,96.5,6.33,1.537,2.248e-06,1.901e-06
2018,17.48,81.9,96.8,6.30,1.650,2.405e-06,2.036e-06
2019,17.45,84.3,99.1,6.12,1.647,2.331e-06,1.983e-06
2020,18.72,93.5,106.1,5.52,1.781,2.273e-06,2.003e-06
2021,21.44,96.4,109.0,5.36,2.074,2.569e-06,2.271e-06
2022,22.71,94.7,107.2,5.45,2.215,2.791e-06,2.466e-06


(b) generalised rho, mean 2017-2022: k_Grenoble = 2.436e-06 1/m (range 2.25e-06 - 2.79e-06)
(c) direct 2017->2022 (alpha=0.17, +13,121 m): k_Grenoble = 2.537e-06 1/m


,method,k_per_m,k_per_km,ratio_to_k_Paris,ratio_to_k_France
0,"ρ rule, 11 % (≈2010)",2.044e-06,0.00204,8.8,862
1,generalised ρ (OSM 2017–22),2.436e-06,0.00244,10.5,1027
2,direct Grenoble 2017–22,2.537e-06,0.00254,11.0,1070


## 5. Replay of the MLCC lookup with an upgrade log

The **same** MLCC code, costs and 1000-state cumulative budget grid as step 5 (`cyclelts.optimization`) are run
again, and every upgraded edge is recorded with its previous LTS and its French facility class. The replay is
checked against the saved lookup table. The log is cached in `mlcc_upgrade_log.csv` (≈ 5 min the first time).

In [6]:
from cyclelts.mode_share import GRENOBLE_MODEL
from cyclelts.optimization import UpgradeProblem, COSTS_LOOKUP, mlcc_lookup

G0 = config.grenoble_graph(2026)
problem = UpgradeProblem(G0, COSTS_LOOKUP, GRENOBLE_MODEL)      # identical MLCC + costs as step 5
B_MAX = problem.full_cost_all
N_POINTS, B_END = 1000, 0.5 * B_MAX
budget_grid = np.linspace(0.0, B_END, N_POINTS)

LOG_CSV = OUT / 'mlcc_upgrade_log.csv'
if LOG_CSV.exists():
    log = pd.read_csv(LOG_CSV)
else:
    logs = []
    def record(i, st):
        for r in st['log']:
            d0 = G0[r['u']][r['v']]
            logs.append(dict(step=i, budget_allocated_eur=float(budget_grid[i]), from_lts=r['from_lts'],
                             facility_before=facility_class(d0), km=km_of(d0), cost_eur=r['cost_eur']))
    mlcc_lookup(problem, budget_grid, on_step=record, checkpoint_every=200)
    log = pd.DataFrame(logs)
    log.to_csv(LOG_CSV, index=False)

lookup = pd.read_csv(LOOKUP_DIR / 'grenoble_lookup_with_mode_share.csv')
chk = log.groupby('step').cost_eur.sum().cumsum().reindex(range(1, N_POINTS)).ffill().fillna(0)
err = np.abs(chk.values - lookup.actual_spent_eur.values[1:]).max()
print(f'Upgrade log: {len(log)} edges, {log.km.sum():.1f} km, €{log.cost_eur.sum():,.0f}')
print(f'Max difference with the saved lookup (cumulative spent): €{err:.4f}')
assert err < 1.0, 'Replay does not match the lookup table'


Upgrade log: 3160 edges, 183.9 km, €38,595,135
Max difference with the saved lookup (cumulative spent): €0.0000


In [7]:
log['new_piste_equivalent'] = ~log.facility_before.isin(['piste', 'bande'])
print('What MLCC upgrades over the whole lookup (0 -> 0.5 B_max):')
mix = log.pivot_table(index='facility_before', columns='from_lts', values='km', aggfunc='sum', fill_value=0)
mix['total_km'] = mix.sum(axis=1)
mix['cost_MEUR'] = log.groupby('facility_before').cost_eur.sum() / 1e6
display(mix.reindex([c for c in CLASS_ORDER if c in mix.index]).round(2))

steps = pd.DataFrame({'step': range(N_POINTS)})
per_step = log.groupby('step').agg(
    new_pb_km=('km', lambda s: s[log.loc[s.index, 'new_piste_equivalent']].sum()),
    convert_km=('km', lambda s: s[~log.loc[s.index, 'new_piste_equivalent']].sum()),
    upgraded_km=('km', 'sum'))
curve = lookup[['index', 'budget_allocated_eur', 'actual_spent_eur', 'predicted_share_percent']].rename(
    columns={'index': 'step'}).merge(per_step, on='step', how='left').fillna(0)
for c in ['new_pb_km', 'convert_km', 'upgraded_km']:
    curve['cum_' + c] = curve[c].cumsum()
curve['mlcc_eur_per_new_pb_m'] = curve.actual_spent_eur / (1000 * curve.cum_new_pb_km).replace(0, np.nan)
curve.to_csv(OUT / 'mlcc_curve_with_piste_bande_equivalent_km.csv', index=False)
display(curve.iloc[::111][['budget_allocated_eur', 'actual_spent_eur', 'predicted_share_percent',
                           'cum_new_pb_km', 'cum_convert_km', 'cum_upgraded_km', 'mlcc_eur_per_new_pb_m']].round(2))

What MLCC upgrades over the whole lookup (0 -> 0.5 B_max):


from_lts,2,3,4,total_km,cost_MEUR
facility_before,,,,,
bande,37.49,0.15,0.00,37.64,7.56
voie verte,10.05,0.00,0.00,10.05,2.01
zone apaisée,1.12,0.00,0.00,1.12,0.22
partage/bus,11.05,1.42,0.07,12.54,2.84
trafic mixte,116.36,5.56,0.64,122.57,25.96


,budget_allocated_eur,actual_spent_eur,predicted_share_percent,cum_new_pb_km,cum_convert_km,cum_upgraded_km,mlcc_eur_per_new_pb_m
0,0.00,0.00,22.82,0.00,0.00,0.00,NaN
111,4288414.94,4287333.23,29.56,11.24,3.33,14.57,381.42
222,8576829.87,8558224.95,32.54,27.03,7.43,34.46,316.65
333,12865244.81,12834474.77,34.11,43.59,11.52,55.11,294.43
444,17153659.74,17145426.44,35.17,60.89,15.78,76.67,281.59
555,21442074.68,21435876.71,36.19,77.87,20.25,98.12,275.29
666,25730489.62,25719217.67,37.16,94.47,25.07,119.54,272.26
777,30018904.55,30010924.46,38.08,112.01,28.98,141.00,267.93
888,34307319.49,34306375.11,38.96,129.63,32.84,162.47,264.65
999,38595734.43,38595135.06,39.79,146.28,37.64,183.92,263.85


*new piste-equivalent km* = upgraded edges that were **not** piste/bande before (calm streets, mixed traffic, bus
lanes, LTS 5 roads): after the upgrade to LTS 1 they become protected facilities, i.e. new linear in the colleague's
sense. *convert km* = bande → piste (or piste repair) upgrades: they cost money and raise LTS, but add **no** km in the
colleague's metric. This is exactly the information that the first-order model cannot see.

## 6. Same axes: share vs budget, share vs new piste + bande km

Both curves start from the same 2026 point (MLCC baseline $P_0$ = 22.82 %; observed 2022 = 22.71 %).
The colleague's curves use $P_b^\infty=0.5$, the three $k^{Grenoble}$ of section 4 and $c^d$ = 400 €/m.
(If the 400 is really €/m², multiply by the lane width — about 2 m → 800 €/m; set `COST_EUR_PER_M`.)

In [8]:
P0_2026 = curve.predicted_share_percent.iloc[0] / 100

def first_order(x_km, k, P_inf, P0=P0_2026):
    return P0 + (P_inf - P0) * (1 - np.exp(-k * np.asarray(x_km, float)))

B = curve.actual_spent_eur.values
fig1, fig2 = go.Figure(), go.Figure()
fig1.add_scatter(x=B / 1e6, y=curve.predicted_share_percent, name='LTS / MLCC lookup', line=dict(width=3, color='black'))
fig2.add_scatter(x=curve.cum_new_pb_km, y=curve.predicted_share_percent, name='LTS / MLCC lookup',
                 line=dict(width=3, color='black'))
KCOL = {'ρ rule, 11 % (≈2010)': '#d62728', 'generalised ρ (OSM 2017–22)': '#ff7f0e', 'direct Grenoble 2017–22': '#2ca02c'}
km_budget = B / (COST_EUR_PER_M * 1000)
for label, k in COLLEAGUE_K.items():
    fig1.add_scatter(x=B / 1e6, y=100 * first_order(km_budget, k, P_INF), name=f'first-order, k {label}',
                     line=dict(color=KCOL[label]))
    fig2.add_scatter(x=curve.cum_new_pb_km, y=100 * first_order(curve.cum_new_pb_km, k, P_INF),
                     name=f'first-order, k {label}', line=dict(color=KCOL[label]))
fig1.update_layout(template='plotly_white', height=520, title='Budget vs bike mode share (2026 start)',
                   xaxis_title='Spent budget (M€)', yaxis_title='Bike mode share (%)')
fig2.update_layout(template='plotly_white', height=520, title='New piste + bande (equivalent) km vs bike mode share',
                   xaxis_title='Added piste + bande km (MLCC: upgraded non-piste/bande edges)',
                   yaxis_title='Bike mode share (%)')
fig1.show(); fig2.show()

## 7. Equivalent $k$ of the LTS/MLCC curve

The first-order equation is fitted to the MLCC curve with $P^\infty$ fixed (as in the scenarios) and also free.
Two versions: $u$ = new piste-equivalent km (comparable with the colleague's $k$) and $u$ = budget/400 €/m
(comparable cost basis). A **local** $k$, $k_{loc}(x) = -\frac{d}{dx}\ln(P^\infty-P)$, shows where the LTS placement
makes each km more or less effective than a constant $k$.

In [9]:
def fit_k(x, y, P_inf=None):
    y = np.asarray(y, float); x = np.asarray(x, float)
    if P_inf is None:
        (k, Pi), _ = curve_fit(lambda xx, k, Pi: first_order(xx, k, Pi), x, y, p0=[0.01, 0.45],
                               bounds=([1e-7, y.max() + 1e-4], [10, 1.0]))
        return k, Pi
    m = y < P_inf - 1e-6
    (k,), _ = curve_fit(lambda xx, k: first_order(xx, k, P_inf), x[m], y[m], p0=[0.01], bounds=([1e-8], [10]))
    return k, P_inf

y = curve.predicted_share_percent.values / 100
x_km = curve.cum_new_pb_km.values
x_cost_km = curve.actual_spent_eur.values / (COST_EUR_PER_M * 1000)
frows = []
for name, P_inf in [('P_inf 0.50 (Optimistic = 50% cap of the logit)', 0.50),
                    ('P_inf 0.35 (Intermediate)', 0.35), ('P_inf free', None)]:
    for xname, xx in [('new piste-equivalent km', x_km), ('budget / 400 €/m', x_cost_km)]:
        k, Pi = fit_k(xx, y, P_inf)
        pred = first_order(xx, k, Pi)
        m = y < Pi
        frows.append(dict(P_inf_case=name, u_definition=xname, k_per_km=k, P_inf=Pi,
                          kappa_kL=k * summary.loc[2026, 'piste_bande_km'],
                          rmse_pp=100 * np.sqrt(np.mean((pred[m] - y[m]) ** 2))))
fits = pd.DataFrame(frows)
display(fits.round(5))
fits.to_csv(OUT / 'equivalent_k_of_mlcc_curve.csv', index=False)

# local k along the curve (P_inf = 0.5), smoothed over 25 states
Pi = 0.5
loc = pd.DataFrame({'x': x_km, 'g': -np.log(Pi - y)}).groupby('x', as_index=False).g.max()
loc = loc[loc.x > 0]
k_loc = pd.Series(np.gradient(loc.g.values, loc.x.values)).rolling(25, center=True, min_periods=5).median()
fig = go.Figure()
fig.add_scatter(x=loc.x, y=k_loc, name='local k of MLCC (P∞ = 0.5)', line=dict(color='black'))
for label, kg in COLLEAGUE_K.items():
    fig.add_hline(y=kg, line_dash='dash', line_color=KCOL[label], annotation_text=f'k {label}')
fig.update_layout(template='plotly_white', height=420, yaxis_type='log', title='Local k along the MLCC curve',
                  xaxis_title='Added piste + bande equivalent km', yaxis_title='k (1/km)')
fig.show()

,P_inf_case,u_definition,k_per_km,P_inf,kappa_kL,rmse_pp
0,P_inf 0.50 (Optimistic = 50% cap of the logit),new piste-equivalent km,0.00870,0.50000,0.96454,2.54374
1,P_inf 0.50 (Optimistic = 50% cap of the logit),budget / 400 €/m,0.01273,0.50000,1.41088,2.09112
2,P_inf 0.35 (Intermediate),new piste-equivalent km,0.06817,0.35000,7.55793,0.77907
3,P_inf 0.35 (Intermediate),budget / 400 €/m,0.07939,0.35000,8.80226,0.63096
4,P_inf free,new piste-equivalent km,0.02523,0.39801,2.79680,1.31658
5,P_inf free,budget / 400 €/m,0.03433,0.39801,3.80617,0.94634


## 8. Order-of-magnitude summary

In [10]:
def budget_for_gain(B_eur, share, gain_pp):
    target = share[0] + gain_pp
    idx = np.argmax(share >= target)
    return B_eur[idx] if share[idx] >= target else np.nan

rows = []
share_mlcc = curve.predicted_share_percent.values
for gain in (2.5, 5.0, 10.0):
    r = dict(gain_pp=gain,
             MLCC_budget_MEUR=budget_for_gain(B, share_mlcc, gain) / 1e6,
             MLCC_new_pb_km=np.interp(budget_for_gain(B, share_mlcc, gain), B, x_km))
    for label, k in COLLEAGUE_K.items():
        frac = (gain / 100) / (P_INF - P0_2026)
        km = -np.log(1 - frac) / k
        r[f'{label} | km'] = km
        r[f'{label} | M€'] = km * COST_EUR_PER_M * 1000 / 1e6
    rows.append(r)
oom = pd.DataFrame(rows).set_index('gain_pp').T
display(oom.round(2))
oom.to_csv(OUT / 'order_of_magnitude_summary.csv')

slope_mlcc = (share_mlcc[np.argmax(B >= 5e6)] - share_mlcc[0]) / (B[np.argmax(B >= 5e6)] / 1e6)
print(f'MLCC average slope over the first 5 M€: {slope_mlcc:.3f} pp per M€')
for label, k in COLLEAGUE_K.items():
    print(f'first-order, k {label:28s}: {100*k*(P_INF-P0_2026)/(COST_EUR_PER_M*1000/1e6):.3f} pp per M€ (initial)')

gain_pp,2.5,5.0,10.0
MLCC_budget_MEUR,0.18,2.93,9.15
MLCC_new_pb_km,0.44,7.12,29.41
"ρ rule, 11 % (≈2010) | km",47.21,99.46,224.43
"ρ rule, 11 % (≈2010) | M€",18.88,39.78,89.77
generalised ρ (OSM 2017–22) | km,39.61,83.45,188.30
generalised ρ (OSM 2017–22) | M€,15.84,33.38,75.32
direct Grenoble 2017–22 | km,38.03,80.12,180.80
direct Grenoble 2017–22 | M€,15.21,32.05,72.32


MLCC average slope over the first 5 M€: 1.493 pp per M€
first-order, k ρ rule, 11 % (≈2010)        : 0.139 pp per M€ (initial)
first-order, k generalised ρ (OSM 2017–22) : 0.166 pp per M€ (initial)
first-order, k direct Grenoble 2017–22     : 0.172 pp per M€ (initial)


## 9. Harmonised comparison: what share of the growth is attributed to lanes?

The colleague's $k$ only explains a fraction $\alpha$ of the observed growth ($k^{Paris}$ is reproduced with
α ≈ 0.17, section 4; the book table uses 0.17–0.25); the rest is due to other
factors. The LTS logit was fitted across 11 cities and years, so it attributes to the network everything correlated with
it. Its **implied $\alpha$** on Grenoble's own history is
$$\alpha_{LTS}=\frac{P^{LTS}(2022)-P^{LTS}(2017)}{P^{obs}(2022)-P^{obs}(2017)}$$
using the logit applied to the 2017 and 2022 LTS networks. The MLCC curve is then also shown with the colleague's
$\alpha$ applied, $P = P_0 + \frac{\alpha}{\alpha_{LTS}}(P_{MLCC}-P_0)$, i.e. only the lane-driven part, like their $k$.

In [11]:
met = pd.read_csv(LTS_DIR / 'grenoble_2017_2022_2026_SIMPLIFIED_metrics.csv').set_index('year')
met['logit_share_pct'] = GRENOBLE_MODEL.share_from_metrics(met.normalized_network_length, met.weighted_lcc_normalized)
hist = met[['normalized_network_length', 'weighted_lcc_normalized', 'logit_share_pct']].join(
    summary[['bike_share_pct', 'piste_bande_km']])
display(hist.round(3))
d_model = hist.loc[2022, 'logit_share_pct'] - hist.loc[2017, 'logit_share_pct']
d_obs = hist.loc[2022, 'bike_share_pct'] - hist.loc[2017, 'bike_share_pct']
ALPHA_LTS = d_model / d_obs
print(f'2017->2022: observed +{d_obs:.2f} pp, LTS logit +{d_model:.2f} pp  ->  implied alpha_LTS = {ALPHA_LTS:.2f}')

fig = go.Figure()
fig.add_scatter(x=B / 1e6, y=curve.predicted_share_percent, name='MLCC (as in lookup)', line=dict(color='black', width=3))
hrows = []
dashes = ['dot', 'dash', 'dashdot']
for j, a in enumerate([ALPHA_COLLEAGUE, 0.21, 0.25]):
    adj = 100 * P0_2026 + (a / ALPHA_LTS) * (curve.predicted_share_percent - 100 * P0_2026)
    fig.add_scatter(x=B / 1e6, y=adj, name=f'MLCC, lane part only (α={a:.2f})', line=dict(color='grey', dash=dashes[j]))
    hrows.append(dict(curve=f'MLCC × α/α_LTS, α={a:.2f}',
                      slope_first_5MEUR_pp_per_MEUR=slope_mlcc * a / ALPHA_LTS,
                      gain_at_10MEUR_pp=float(np.interp(10e6, B, adj)) - 100 * P0_2026,
                      gain_at_full_lookup_pp=float(adj.iloc[-1]) - 100 * P0_2026))
for label, k in COLLEAGUE_K.items():
    fo = 100 * first_order(B / (COST_EUR_PER_M * 1000), k, P_INF)
    fig.add_scatter(x=B / 1e6, y=fo, name=f'first-order, k {label}', line=dict(color=KCOL[label]))
    hrows.append(dict(curve=f'first-order, k {label}',
                      slope_first_5MEUR_pp_per_MEUR=(float(np.interp(5e6, B, fo)) - 100 * P0_2026) / 5,
                      gain_at_10MEUR_pp=float(np.interp(10e6, B, fo)) - 100 * P0_2026,
                      gain_at_full_lookup_pp=float(fo[-1]) - 100 * P0_2026))
fig.update_layout(template='plotly_white', height=520, title='Harmonised: only the lane-driven part of the share',
                  xaxis_title='Spent budget (M€)', yaxis_title='Bike mode share (%)')
fig.show()
harm = pd.DataFrame(hrows)
display(harm.round(3))
harm.to_csv(OUT / 'harmonised_alpha_comparison.csv', index=False)

,normalized_network_length,weighted_lcc_normalized,logit_share_pct,bike_share_pct,piste_bande_km
year,,,,,
2017,0.930,0.324,18.091,16.385,81.569
2018,0.939,0.326,18.389,17.480,81.903
2019,0.965,0.335,19.327,17.449,84.330
2020,0.968,0.359,20.248,18.721,93.488
2021,0.989,0.372,21.231,21.441,96.354
2022,0.993,0.375,21.432,22.711,94.690
2026,1.000,0.407,22.820,NaN,110.868


2017->2022: observed +6.33 pp, LTS logit +3.34 pp  ->  implied alpha_LTS = 0.53


,curve,slope_first_5MEUR_pp_per_MEUR,gain_at_10MEUR_pp,gain_at_full_lookup_pp
0,"MLCC × α/α_LTS, α=0.17",0.492,3.421,5.591
1,"MLCC × α/α_LTS, α=0.21",0.594,4.129,6.748
2,"MLCC × α/α_LTS, α=0.25",0.707,4.916,8.034
3,"first-order, k ρ rule, 11 % (≈2010)",0.137,1.354,4.865
4,"first-order, k generalised ρ (OSM 2017–22)",0.163,1.606,5.694
5,"first-order, k direct Grenoble 2017–22",0.170,1.671,5.902
